In [1]:
!pip install pinecone

In [2]:
!pip install langchain langchain-core langchain-openai langchain-community \
            langchain-chroma chromadb pymupdf python-dotenv langchain-huggingface sentence-transformers

In [18]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="qwen3-4b",
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9409.39it/s]


In [19]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("C:\\Users\\ahmed\\Rag_project_software\\nike_football_catalog.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # small — see note below
    chunk_overlap=50,
    separators=["\n\n", "\n", " "],
)
chunks = splitter.split_documents(docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 178


In [20]:
#only once
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./nike_chroma_db",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [21]:
# what happens after the first run
vectorstore = Chroma(
    persist_directory="./nike_chroma_db",
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [22]:
from langchain.tools import tool

@tool
def nike_search(query: str) -> str:
    """Search the Nike product catalog. Use for questions about shirts,
    boots, shorts, club kits, prices, or availability."""
    docs = retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs)

@tool
def filter_by_price(query: str, max_price: float) -> str:
    """Search Nike products and return only those under a given price in GBP."""
    docs = retriever.invoke(query)
    results = []
    for d in docs:
        text = d.page_content
        import re
        prices = re.findall(r'£([\d.]+)', text)
        if prices and all(float(p) <= max_price for p in prices):
            results.append(text)
    return "\n\n".join(results) if results else "No products found under that price."

In [23]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def run_nike_agent(question: str):
    tools = [nike_search, filter_by_price]
    tools_dict = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = [
        SystemMessage(content=(
            "You are a Nike UK football product assistant. "
            "Use the nike_search tool to find products. "
            "Use filter_by_price when the user mentions a budget. "
            "Always mention the price in GBP in your final answer."
        )),
        HumanMessage(content=question),
    ]

    for i in range(1, 10):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f"\nAnswer: {response.content}")
            return response.content

        for tc in response.tool_calls:
            result = tools_dict[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

run_nike_agent("Show me all the prices of the arsenal merch")


Answer: 

Based on my search for Arsenal merchandise, I found some products in the Nike catalog, though they appear to be from other teams rather than Arsenal:

**Available Products and Prices:**

1. **Stade Toulousain X Toulouse F.C. Unisex Nike Capitolium Pique Polo** - £34.99
2. **Tottenham Hotspur 2025/26 Match Away Men's Nike Dri-FIT ADV Football Authentic Shirt** - £124.99
3. **Tottenham Hotspur 2025/26 Match Home Men's Nike Dri-FIT ADV Football Authentic Shirt** - £124.99
4. **Tottenham Hotspur 2025/26 Stadium** - Price not specified

Please note that these search results don't appear to include actual Arsenal merchandise. The products shown are from Tottenham Hotspur and Stade Toulousain instead. If you're specifically looking for Arsenal FC items, you might want to try a more specific search term like "Arsenal FC" or "Arsenal kit" to get more accurate results.


'\n\nBased on my search for Arsenal merchandise, I found some products in the Nike catalog, though they appear to be from other teams rather than Arsenal:\n\n**Available Products and Prices:**\n\n1. **Stade Toulousain X Toulouse F.C. Unisex Nike Capitolium Pique Polo** - £34.99\n2. **Tottenham Hotspur 2025/26 Match Away Men\'s Nike Dri-FIT ADV Football Authentic Shirt** - £124.99\n3. **Tottenham Hotspur 2025/26 Match Home Men\'s Nike Dri-FIT ADV Football Authentic Shirt** - £124.99\n4. **Tottenham Hotspur 2025/26 Stadium** - Price not specified\n\nPlease note that these search results don\'t appear to include actual Arsenal merchandise. The products shown are from Tottenham Hotspur and Stade Toulousain instead. If you\'re specifically looking for Arsenal FC items, you might want to try a more specific search term like "Arsenal FC" or "Arsenal kit" to get more accurate results.'